# ChainScore — Time Series Score Evolution

Reconstructs credit scores at monthly snapshots over a 12-month window.
Each snapshot filters transactions to only those available at that point in time,
simulating what the model would have predicted historically.

This enables:
- Monitoring score drift for counterparty risk surveillance
- Identifying wallets that deteriorated before liquidation
- Backtesting alert thresholds

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
from pathlib import Path
from src.features.builder import build_feature_matrix

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white', 'font.family': 'monospace'})

In [ ]:
# Load models
lr  = joblib.load('../models/logistic_regression.pkl')
feature_cols = pd.read_json('../models/feature_columns.json', typ='series').tolist()

NORMAL_TXS  = Path('../data/raw/wallets/normal_txs.parquet')
TOKEN_TXS   = Path('../data/raw/wallets/token_txs.parquet')
DEFAULTS    = Path('../data/raw/aave_v2_liquidations.parquet')
NON_DEFAULTS= Path('../data/raw/non_default_cohort.parquet')

def pd_to_score(p): return int(round((1 - p) * 1000))

def score_at_date(cutoff: pd.Timestamp, sample_wallets: list[str] | None = None) -> pd.DataFrame:
    """Build features at cutoff date and return scores."""
    tmp_path = Path(f'../data/processed/ts_{cutoff.strftime("%Y%m")}.parquet')
    df = build_feature_matrix(
        NORMAL_TXS, TOKEN_TXS, DEFAULTS, NON_DEFAULTS,
        output_path=tmp_path, max_date=cutoff
    )
    if sample_wallets:
        df = df[df['wallet'].isin(sample_wallets)]
    X = df[feature_cols]
    pds = lr.predict_proba(X)[:, 1]
    df = df[['wallet', 'label']].copy()
    df['score'] = [pd_to_score(p) for p in pds]
    df['pd']    = pds
    df['date']  = cutoff
    tmp_path.unlink(missing_ok=True)  # clean up temp file
    return df

print('Setup complete.')

In [ ]:
# Define monthly snapshots — 12 months ending at the train/test cutoff (Apr 2023)
cutoffs = pd.date_range(end='2023-04-01', periods=12, freq='MS', tz='UTC')
print('Snapshot dates:', [str(c.date()) for c in cutoffs])
print('NOTE: This cell takes ~5-10 min — each snapshot rebuilds features for all wallets.')

In [ ]:
# Run snapshots — collect portfolio-level stats at each date
# For a quick run, use a random sample of wallets
base_df = pd.read_parquet('../data/processed/feature_matrix.parquet')
SAMPLE_N = 500  # increase for more precision, decrease for speed
sample_wallets = base_df.sample(n=min(SAMPLE_N, len(base_df)), random_state=42)['wallet'].tolist()

snapshots = []
for cutoff in cutoffs:
    print(f'Scoring at {cutoff.date()}...')
    snap = score_at_date(cutoff, sample_wallets=sample_wallets)
    snapshots.append(snap)

ts = pd.concat(snapshots, ignore_index=True)
print(f'Done. {len(ts):,} wallet-date observations.')

In [ ]:
# Portfolio-level time series
portfolio_ts = ts.groupby('date').agg(
    mean_score=('score', 'mean'),
    median_score=('score', 'median'),
    pct_high_risk=('score', lambda x: (x < 500).mean()),
    pct_very_high=('score', lambda x: (x < 300).mean()),
    mean_pd=('pd', 'mean'),
    var_95=('pd', lambda x: np.percentile(x, 95)),
).reset_index()

print(portfolio_ts[['date', 'mean_score', 'pct_high_risk', 'mean_pd']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

dates = portfolio_ts['date']

# Crisis markers
crises = [
    ('2022-05-01', 'LUNA', '#dc2626'),
    ('2022-11-01', 'FTX',  '#f59e0b'),
    ('2023-03-01', 'USDC', '#8b5cf6'),
]

def add_crises(ax):
    for date, label, color in crises:
        d = pd.Timestamp(date, tz='UTC')
        if dates.min() <= d <= dates.max():
            ax.axvline(d, color=color, linestyle='--', linewidth=1, alpha=0.7)
            ax.text(d, ax.get_ylim()[1], label, color=color, fontsize=7, rotation=90, va='top', ha='right')

# 1. Mean score over time
ax = axes[0, 0]
ax.plot(dates, portfolio_ts['mean_score'], 'b-o', markersize=4, label='Mean score')
ax.fill_between(dates, portfolio_ts['median_score'], portfolio_ts['mean_score'], alpha=0.15, color='blue')
ax.set_ylabel('ChainScore')
ax.set_title('Portfolio Mean Score')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
add_crises(ax)

# 2. High-risk concentration
ax = axes[0, 1]
ax.plot(dates, portfolio_ts['pct_high_risk'] * 100, 'r-o', markersize=4, label='High risk')
ax.plot(dates, portfolio_ts['pct_very_high'] * 100, 'r--s', markersize=4, alpha=0.6, label='Very high risk')
ax.set_ylabel('% of wallets')
ax.set_title('High-Risk Concentration Over Time')
ax.legend(fontsize=8)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
add_crises(ax)

# 3. Mean PD
ax = axes[1, 0]
ax.plot(dates, portfolio_ts['mean_pd'], 'g-o', markersize=4)
ax.set_ylabel('Avg PD')
ax.set_title('Portfolio Average Probability of Default')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
add_crises(ax)

# 4. VaR 95%
ax = axes[1, 1]
ax.plot(dates, portfolio_ts['var_95'], 'orange', marker='o', markersize=4)
ax.set_ylabel('PD at 95th pct')
ax.set_title('Portfolio VaR 95% (Tail Risk)')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
add_crises(ax)

fig.suptitle('ChainScore Time Series — Portfolio Risk Evolution (12-month window)', fontsize=12)
plt.tight_layout()
plt.savefig('../reports/figures/time_series_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reports/figures/time_series_scores.png')